# Mamíferos de Colombia — Semana 10: Defensa técnica del modelo y regularización L1/L2

**Integrantes:** Jaime Andrés Ortega Cadena · David Alejandro Muñoz Carmona
**Proyecto de impacto regional:** clasificación de 40 mamíferos de Colombia en fotos de cámaras trampa, para apoyar el monitoreo de fauna de Cortolima (Tolima).

## Qué se entrega en este cuaderno
| Sección | Contenido |
|---|---|
| 0–3 | Misma configuración, ingesta, auditoría, división por registro GBIF y lectura de imágenes que el entrenamiento del proyecto |
| 4 | Características de cada foto con una red base congelada (MobileNetV3-Large, ImageNet) |
| 5 | **Las tres variantes de regularización:** solo Dropout · Dropout + L2 · Dropout + L1 |
| 6 | **Curvas de pérdida** de cada variante (entrenamiento vs. validación) |
| 7 | **Tabla comparativa** (validación, prueba, sobreajuste, pesos en cero) |
| 8 | Sensibilidad a λ y decisión final |
| 9 | Genera la **tabla de decisiones técnicas en PDF (1 página)** |

## Diseño del experimento (controlado)
Para que la única diferencia entre variantes sea la regularización:
- Las **mismas fotos** y la **misma división** (entrenamiento 17.960 · validación 2.424 · prueba 2.458, separadas por registro GBIF).
- Las **mismas características**: la red base congelada se calcula una sola vez.
- La **misma cabeza** que el modelo final del proyecto: `Dropout 0,4 → Densa 512 ReLU → Dropout 0,3 → Salida 40`.
- La **misma semilla**, el mismo optimizador (Adam 0,001), 40 épocas, lote 256 y el mismo **λ = 0,0001** que usa el modelo real.
- Solo cambia la penalización de los pesos de las capas densas:
  - **Solo Dropout:** pérdida = entropía cruzada.
  - **Dropout + L2:** pérdida = entropía cruzada + λ·Σw² → pesos pequeños, pero no cero.
  - **Dropout + L1:** pérdida = entropía cruzada + λ·Σ|w| → muchos pesos exactamente en cero.

Se agrega una corrida de **referencia sin Dropout ni L1/L2**, para medir cuánto ayuda el Dropout.

**Regla de decisión (fijada antes de ver los resultados):** se elige la variante con mayor exactitud en **validación**. Si dos variantes quedan a menos de 0,5 puntos, se prefiere la de **menor brecha entre entrenamiento y validación** (menos sobreajuste). La **prueba** solo se usa al final para confirmar.

**Cómo ejecutar:** panel derecho → *Input* → dataset `mamiferos-colombia-40` · *Accelerator* → GPU T4 · luego *Run All* (unos 5 minutos).

---
# 0–3. Configuración, ingesta, auditoría, división y lectura (idénticas al entrenamiento del proyecto)

In [ ]:
# 0.1 CONFIGURACIÓN (la misma del proyecto; aquí solo se usan datos, lote y semilla)
import os, sys, io, json, time, math, random, shutil, zipfile, hashlib, logging, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
warnings.filterwarnings("ignore")

EN_KAGGLE = Path("/kaggle/input").exists()

# ---------- Datos ----------
FRAC_VAL, FRAC_TEST = 0.15, 0.15       # validación y prueba: SOLO fotos de Colombia, separadas por registro (gbifID)
MAX_TRAIN_POR_CLASE = 1200             # tope en entrenamiento para que la ardilla colirroja (2.452) no domine

# ---------- Modelo profesor (Transfer Learning) ----------
MODELO_BASE = "EfficientNetV2S"
IMG_SIZE = 260
BATCH = 32
DENSA = 512                            # neuronas de la capa densa antes de la salida
DROPOUT_1, DROPOUT_2 = 0.40, 0.30      # anti-overfitting
REGULARIZACION, FACTOR_REG = "l2", 1e-4   # "l1", "l2", "l1_l2" o None  (penaliza los pesos grandes)
LABEL_SMOOTHING = 0.10

# ---------- Entrenamiento ----------
EPOCAS_ETAPA1 = 10                     # etapa 1: base congelada, solo se entrena la cabeza
EPOCAS_TOTALES = 60                    # etapa 2: ajuste fino de la parte alta de la base (épocas 11 a 60 = 50 épocas)
LR_ETAPA1, LR_ETAPA2 = 1e-3, 1e-4      # optimizador Adam
FRACCION_AJUSTE_FINO = 0.60            # se descongela el 60 % final de las capas de la base
PACIENCIA = 12                         # EarlyStopping: para si val_accuracy no mejora en 12 épocas
EPOCA_MINIMA = 50                      # EarlyStopping no puede detener el entrenamiento antes de la época 50
META_ACCURACY = 0.94

MODO_PRUEBA = False                    # True = corrida corta de demostración (pocos lotes, 2 épocas) para verificar el pipeline
SEED = 42
random.seed(SEED); np.random.seed(SEED)

print("Kaggle:", EN_KAGGLE, "| modelo:", MODELO_BASE, f"{IMG_SIZE}px | épocas: {EPOCAS_ETAPA1} + {EPOCAS_TOTALES - EPOCAS_ETAPA1} = {EPOCAS_TOTALES}",
      "| MODO_PRUEBA" if MODO_PRUEBA else "")

In [ ]:
# 0.2 Entorno: GPU, TensorFlow/Keras, carpetas de salida y LOG
import tensorflow as tf
import keras
from keras import layers, regularizers
from IPython.display import display

BASE_SALIDA = Path("/kaggle/working") if EN_KAGGLE else Path.cwd()
SALIDA = BASE_SALIDA / "regularizacion"   # salidas de este experimento
EVI = SALIDA / "evidencia"
for p in [SALIDA, EVI, SALIDA / "logs"]:
    p.mkdir(parents=True, exist_ok=True)

log = logging.getLogger("ENTRENAMIENTO"); log.setLevel(logging.INFO); log.handlers.clear(); log.propagate = False
_fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%Y-%m-%d %H:%M:%S")
_fh = logging.FileHandler(SALIDA / "logs" / f"entrenamiento_{datetime.now():%Y%m%d_%H%M%S}.log", encoding="utf-8")
_sh = logging.StreamHandler(sys.stdout)
for h in (_fh, _sh):
    h.setFormatter(_fmt); log.addHandler(h)

GPUS = tf.config.list_physical_devices("GPU")
if GPUS:
    keras.mixed_precision.set_global_policy("mixed_float16")     # ~2x más rápido en GPU T4
keras.utils.set_random_seed(SEED)
log.info("=" * 70)
log.info(f"SESIÓN DE ENTRENAMIENTO | TensorFlow {tf.__version__} | Keras {keras.__version__} | "
         f"GPU: {[g.name for g in GPUS] or 'NINGUNA'} | precisión: {keras.mixed_precision.global_policy().name}")
log.info(f"Salidas en: {SALIDA}")
if not GPUS:
    print("⚠️  No hay GPU. Panel derecho → Session options → Accelerator: GPU T4 x2 (requiere celular verificado).")

In [ ]:
# 1.1 Ingesta: ubicar el dataset subido a Kaggle (Kaggle descomprime el ZIP solo) y leer metadata.csv
RAIZ = Path("/kaggle/input") if EN_KAGGLE else Path.cwd()
encontrados = sorted(RAIZ.rglob("metadata.csv"))
if encontrados:
    DATOS = encontrados[0].parent
else:                                                  # por si el ZIP no se descomprimió
    zips = sorted(RAIZ.rglob("*.zip"))
    assert zips, "No encuentro el dataset. Panel derecho → Input → Add Input → Your Datasets → mamiferos-colombia-40"
    DATOS = Path("/tmp/datos")
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(DATOS)
meta = pd.read_csv(DATOS / "metadata.csv")
carpeta_img = DATOS / "dataset" if (DATOS / "dataset").exists() else DATOS
meta["ruta"] = meta["archivo"].map(lambda a: str(carpeta_img / a))
log.info(f"Dataset en: {DATOS}")
log.info(f"INGESTA: {len(meta)} imágenes | {meta.clase.nunique()} clases | "
         f"Colombia {int((meta.origen == 'colombia').sum())} | Latinoamérica {int((meta.origen != 'colombia').sum())}")

In [ ]:
# 1.2 Auditoría automática de la ingesta (limpieza antes de entrenar)
def auditar(ruta):
    p = Path(ruta)
    if not p.exists():
        return "no_existe", ""
    data = p.read_bytes()
    try:
        with Image.open(io.BytesIO(data)) as im:
            im.verify()
        with Image.open(io.BytesIO(data)) as im:
            w, h = im.size
    except Exception:
        return "corrupta", ""
    if min(w, h) < 64:
        return "muy_pequena", ""
    return "", hashlib.sha256(data).hexdigest()

t0 = time.time()
res = [auditar(r) for r in meta["ruta"]]
meta["problema"] = [r[0] for r in res]; meta["sha256"] = [r[1] for r in res]
dup = meta["sha256"].ne("") & meta.duplicated("sha256", keep="first")
meta.loc[dup, "problema"] = "duplicado_exacto"
aud = meta["problema"].replace("", "OK").value_counts()
log.info(f"AUDITORÍA ({time.time() - t0:.0f} s): " + " | ".join(f"{k}={v}" for k, v in aud.items()))
display(aud.to_frame("imágenes"))
meta = meta[meta["problema"] == ""].reset_index(drop=True)

CLASES = sorted(meta["clase"].unique())
N_CLASES = len(CLASES)
IDX = {c: i for i, c in enumerate(CLASES)}
meta["y"] = meta["clase"].map(IDX)
NOMBRE = meta.drop_duplicates("clase").set_index("clase")["nombre_comun"].to_dict()
log.info(f"Imágenes válidas para entrenar: {len(meta)} | clases: {N_CLASES}")

In [ ]:
# 2.1 División estratificada por clase y agrupada por registro GBIF
rng = np.random.default_rng(SEED)
meta["split"] = "train"
for c, d in meta.groupby("clase"):
    reg_fuera = set(d.loc[d["origen"] != "colombia", "gbifID"])     # registros de otros países: solo entrenamiento
    col = d[~d["gbifID"].isin(reg_fuera)]
    por_reg = col.groupby("gbifID").size()
    ids = por_reg.index.to_numpy().copy(); rng.shuffle(ids)
    total, acum, asign = int(por_reg.sum()), 0, {}
    for g in ids:
        asign[g] = "test" if acum < total * FRAC_TEST else ("val" if acum < total * (FRAC_TEST + FRAC_VAL) else "train")
        acum += int(por_reg[g])
    meta.loc[col.index, "split"] = col["gbifID"].map(asign).values

# Tope por clase en entrenamiento (se priorizan las fotos de Colombia)
meta["_azar"] = rng.random(len(meta))
tr = meta[meta.split == "train"].sort_values(["clase", "origen", "_azar"],
                                             key=lambda s: (s != "colombia") if s.name == "origen" else s)
sobran = tr.groupby("clase").cumcount() >= MAX_TRAIN_POR_CLASE
meta.loc[tr.index[sobran.values], "split"] = "sin_usar"

# Verificación: ningún registro GBIF puede estar en dos grupos
g = {s: set(meta.loc[meta.split == s, "gbifID"]) for s in ["train", "val", "test"]}
assert not (g["train"] & g["val"]) and not (g["train"] & g["test"]) and not (g["val"] & g["test"]), "¡Fuga de datos!"
assert (meta.loc[meta.split.isin(["val", "test"]), "origen"] == "colombia").all()

tabla_split = pd.crosstab(meta["clase"], meta["split"]).reindex(columns=["train", "val", "test", "sin_usar"], fill_value=0)
display(tabla_split)
tabla_split.to_csv(EVI / "division_por_clase.csv", encoding="utf-8-sig")
meta.drop(columns=["_azar"]).to_csv(SALIDA / "division_dataset.csv", index=False, encoding="utf-8-sig")
cuenta = meta["split"].value_counts()
log.info("DIVISIÓN por gbifID (sin fuga de datos ✓): " + " | ".join(f"{k}={cuenta.get(k, 0)}" for k in ["train", "val", "test", "sin_usar"]))

In [ ]:
# 2.2 Pesos por clase (compensan el desbalance): peso = N / (K · n_clase), limitado entre 0.5 y 5
n_tr = meta[meta.split == "train"].groupby("y").size().reindex(range(N_CLASES), fill_value=1)
PESOS = np.clip(n_tr.sum() / (N_CLASES * n_tr), 0.5, 5.0)
meta["peso"] = meta["y"].map(PESOS)
display(pd.DataFrame({"clase": CLASES, "imagenes_train": n_tr.values, "peso": PESOS.round(2).values})
          .sort_values("peso", ascending=False).head(10))
log.info(f"Pesos por clase: mínimo {PESOS.min():.2f} ({CLASES[int(PESOS.idxmin())]}) | "
         f"máximo {PESOS.max():.2f} ({CLASES[int(PESOS.idxmax())]})")

In [ ]:
# 3.1 Lectura eficiente con tf.data (paralela + prefetch)
AUTOTUNE = tf.data.AUTOTUNE

def leer_imagen(ruta, size):
    img = tf.io.decode_jpeg(tf.io.read_file(ruta), channels=3)
    return tf.image.resize(img, (size, size), antialias=True)          # float32 en 0..255

def hacer_ds(df, size=IMG_SIZE, entrenamiento=False):
    if entrenamiento:
        ds = tf.data.Dataset.from_tensor_slices((df["ruta"].values, df["y"].values.astype("int32"),
                                                 df["peso"].values.astype("float32")))
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(lambda r, y, w: (leer_imagen(r, size), tf.one_hot(y, N_CLASES), w), num_parallel_calls=AUTOTUNE)
    else:
        ds = tf.data.Dataset.from_tensor_slices((df["ruta"].values, df["y"].values.astype("int32")))
        ds = ds.map(lambda r, y: (leer_imagen(r, size), tf.one_hot(y, N_CLASES)), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

TRAIN = meta[meta.split == "train"].reset_index(drop=True)
VAL = meta[meta.split == "val"].reset_index(drop=True)
TEST = meta[meta.split == "test"].reset_index(drop=True)
ds_train, ds_val, ds_test = hacer_ds(TRAIN, entrenamiento=True), hacer_ds(VAL), hacer_ds(TEST)
if MODO_PRUEBA:
    ds_train, ds_val, ds_test = ds_train.take(15), ds_val.take(5), ds_test.take(5)
log.info(f"tf.data listo | train {len(TRAIN)} | val {len(VAL)} | test {len(TEST)} | lote {BATCH} | "
         f"{math.ceil(len(TRAIN) / BATCH)} pasos por época")

---
# 4. Características de cada foto (red base congelada)
Cada foto pasa **una sola vez** por MobileNetV3-Large (pesos de ImageNet, sin su capa final) y se resume en un vector de **960 números**.
Sobre esos vectores se entrenan las variantes. Así, las diferencias entre ellas se deben solo a la regularización y no al azar de la red base.

In [ ]:
# 4.1 Características de entrenamiento, validación y prueba
keras.mixed_precision.set_global_policy("float32")
IMG_EST = 224
base_fija = keras.applications.MobileNetV3Large(input_shape=(IMG_EST, IMG_EST, 3), include_top=False,
                                               weights="imagenet", pooling="avg")
base_fija.trainable = False
t0 = time.time()
FEAT, Y = {}, {}
for g, df in [("train", TRAIN), ("val", VAL), ("test", TEST)]:
    FEAT[g] = base_fija.predict(hacer_ds(df, size=IMG_EST), verbose=0).astype("float32")
    Y[g] = df["y"].values
    if MODO_PRUEBA:
        Y[g] = Y[g][:len(FEAT[g])]
Y1 = {g: keras.utils.to_categorical(Y[g], N_CLASES) for g in Y}
D = FEAT["train"].shape[1]
log.info(f"Características: train {FEAT['train'].shape} | val {FEAT['val'].shape} | test {FEAT['test'].shape} "
         f"| {time.time() - t0:.0f} s")

---
# 5. Las tres variantes de regularización
La cabeza es la misma del modelo final del proyecto. Lo único que cambia es `kernel_regularizer` en las dos capas densas.

Se guarda, además de la pérdida total, la **entropía cruzada pura** (`ce`). La pérdida total de L1 y L2 incluye la penalización λ·Σ|w| o λ·Σw², y por eso no se puede comparar directamente con la de solo Dropout. La entropía cruzada sí es comparable entre todas.

In [ ]:
# 5.1 Cabeza del modelo y entrenamiento de cada variante
from sklearn.metrics import f1_score

LAMBDA = 1e-4                 # el mismo λ del modelo real del proyecto (FACTOR_REG)
EPOCAS_REG, LOTE_REG = 40, 256

def crear_cabeza(reg=None, d1=DROPOUT_1, d2=DROPOUT_2):
    capas = [keras.Input((D,))]
    if d1: capas.append(layers.Dropout(d1, name="dropout_1"))
    capas.append(layers.Dense(DENSA, activation="relu", kernel_regularizer=reg, name="densa"))
    if d2: capas.append(layers.Dropout(d2, name="dropout_2"))
    capas.append(layers.Dense(N_CLASES, kernel_regularizer=reg, name="salida"))
    return keras.Sequential(capas)

class MejorEpoca(keras.callbacks.Callback):
    """Guarda los pesos de la época con mejor exactitud de validación (como en el entrenamiento real)."""
    def __init__(self):
        super().__init__(); self.mejor, self.epoca, self.pesos = -1.0, 0, None
    def on_epoch_end(self, epoch, logs=None):
        if logs["val_accuracy"] > self.mejor:
            self.mejor, self.epoca, self.pesos = logs["val_accuracy"], epoch + 1, self.model.get_weights()

def entrenar(nombre, reg, d1=DROPOUT_1, d2=DROPOUT_2):
    keras.utils.set_random_seed(SEED)
    m = crear_cabeza(reg, d1, d2)
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss=keras.losses.CategoricalCrossentropy(from_logits=True),
              metrics=["accuracy", keras.metrics.CategoricalCrossentropy(from_logits=True, name="ce")])
    cb = MejorEpoca(); t0 = time.time()
    h = m.fit(FEAT["train"], Y1["train"], validation_data=(FEAT["val"], Y1["val"]),
              epochs=2 if MODO_PRUEBA else EPOCAS_REG, batch_size=LOTE_REG, verbose=0, callbacks=[cb])
    m.set_weights(cb.pesos)                                   # se evalúa la mejor época
    hist = pd.DataFrame(h.history); hist.index = hist.index + 1
    # Exactitud de entrenamiento SIN Dropout (modo evaluación), para medir el sobreajuste de forma justa
    acc_tr = float(m.evaluate(FEAT["train"], Y1["train"], batch_size=1024, verbose=0)[1])
    p = m.predict(FEAT["test"], batch_size=1024, verbose=0)
    top3 = np.argsort(-p, axis=1)[:, :3]
    w = np.concatenate([m.get_layer("densa").get_weights()[0].ravel(), m.get_layer("salida").get_weights()[0].ravel()])
    fila = {"variante": nombre,
            "mejor época": cb.epoca,
            "train acc": acc_tr,
            "val acc": cb.mejor,
            "brecha train−val": acc_tr - cb.mejor,
            "val entropía mín.": float(hist["val_ce"].min()),
            "val entropía final": float(hist["val_ce"].iloc[-1]),
            "test acc": float((p.argmax(1) == Y["test"]).mean()),
            "test top-3": float((top3 == Y["test"][:, None]).any(1).mean()),
            "test F1 macro": float(f1_score(Y["test"], p.argmax(1), average="macro")),
            "pesos ≈ 0 (|w|<1e-3)": float((np.abs(w) < 1e-3).mean()),
            "norma L2 de los pesos": float(np.sqrt((w ** 2).sum())),
            "segundos": round(time.time() - t0, 1)}
    log.info(f"   {nombre:<26s} val {fila['val acc']:.4f} (época {cb.epoca:>2d}) | test {fila['test acc']:.4f} | "
             f"brecha {fila['brecha train−val']:+.4f} | pesos≈0 {fila['pesos ≈ 0 (|w|<1e-3)']:.1%}")
    return fila, hist, w

VARIANTES = [("Solo Dropout", None),
             ("Dropout + L2", regularizers.L2(LAMBDA)),
             ("Dropout + L1", regularizers.L1(LAMBDA))]
FILAS, HIST, PESOS_W = [], {}, {}
log.info("=" * 70); log.info(f"REGULARIZACIÓN — 3 variantes | λ = {LAMBDA:g} | Dropout {DROPOUT_1}/{DROPOUT_2}")
for nombre, reg in VARIANTES:
    f, h, w = entrenar(nombre, reg); FILAS.append(f); HIST[nombre] = h; PESOS_W[nombre] = w

# Referencia: la misma cabeza SIN Dropout y SIN L1/L2 (mide cuánto ayuda el Dropout)
REF = "Sin regularización (referencia)"
f, h, w = entrenar(REF, None, d1=0, d2=0); FILA_REF = f; HIST[REF] = h; PESOS_W[REF] = w

---
# 6. Curvas de pérdida
- **Línea continua:** entropía cruzada (comparable entre variantes).
- **Línea punteada:** pérdida total que minimiza el optimizador (entropía cruzada + penalización L1 o L2).
- **Azul** = entrenamiento · **Naranja** = validación · **Punto** = mejor época (la que se guarda).

Si la curva de validación sube mientras la de entrenamiento sigue bajando, el modelo está **memorizando** (sobreajuste).

In [ ]:
# 6.1 Curvas de pérdida de las tres variantes
AZUL, NARANJA = "#1f77b4", "#ff7f0e"
fig, axs = plt.subplots(1, 3, figsize=(19, 5), sharey=True)
for ax, (nombre, _) in zip(axs, VARIANTES):
    h = HIST[nombre]; ep = int(next(f["mejor época"] for f in FILAS if f["variante"] == nombre))
    ax.plot(h.index, h["ce"], color=AZUL, lw=2, label="entrenamiento (entropía)")
    ax.plot(h.index, h["val_ce"], color=NARANJA, lw=2, label="validación (entropía)")
    if nombre != "Solo Dropout":
        ax.plot(h.index, h["loss"], color=AZUL, lw=1.2, ls="--", label="entrenamiento (total con penalización)")
        ax.plot(h.index, h["val_loss"], color=NARANJA, lw=1.2, ls="--", label="validación (total con penalización)")
    ax.scatter([ep], [h.loc[ep, "val_ce"]], color=NARANJA, s=60, zorder=5, edgecolor="black")
    ax.set_title(f"{nombre}  ·  mejor época {ep}", fontsize=13); ax.set_xlabel("época"); ax.grid(alpha=0.3)
axs[0].set_ylabel("pérdida"); axs[0].legend(fontsize=9); axs[2].legend(fontsize=9)
plt.suptitle(f"Curvas de pérdida — misma cabeza, mismo λ = {LAMBDA:g}, misma semilla", fontsize=14)
plt.tight_layout(); plt.savefig(EVI / "curvas_perdida_3_variantes.png", dpi=120); plt.show()

In [ ]:
# 6.2 Las cuatro corridas juntas: pérdida y exactitud de validación, y distribución de los pesos
COLORES = {"Solo Dropout": "#2ca02c", "Dropout + L2": "#1f77b4", "Dropout + L1": "#d62728", REF: "#7f7f7f"}
fig, axs = plt.subplots(1, 3, figsize=(19, 4.8))
for nombre, h in HIST.items():
    ls = ":" if nombre == REF else "-"
    axs[0].plot(h.index, h["val_ce"], ls=ls, lw=2, color=COLORES[nombre], label=nombre)
    axs[1].plot(h.index, h["val_accuracy"], ls=ls, lw=2, color=COLORES[nombre], label=nombre)
    axs[2].hist(PESOS_W[nombre], bins=150, range=(-0.15, 0.15), histtype="step", lw=1.6, color=COLORES[nombre], label=nombre)
axs[0].set_title("Pérdida de validación (entropía cruzada)"); axs[1].set_title("Exactitud de validación")
axs[2].set_title("Distribución de los pesos (escala log)"); axs[2].set_yscale("log"); axs[2].set_xlabel("valor del peso")
for ax in axs[:2]: ax.set_xlabel("época"); ax.grid(alpha=0.3)
axs[0].legend(fontsize=9); plt.tight_layout(); plt.savefig(EVI / "comparacion_validacion_y_pesos.png", dpi=120); plt.show()

---
# 7. Tabla comparativa
- **val acc / test acc:** fotos bien clasificadas en validación y en prueba (fotos que el modelo nunca vio).
- **brecha train−val:** cuánto mejor le va con las fotos que estudió que con las nuevas. Más alta = más sobreajuste.
- **val entropía final − mín.:** cuánto empeoró la pérdida de validación desde su mejor punto. Más alta = más memorización al final.
- **pesos ≈ 0:** fracción de pesos prácticamente apagados. L1 la sube mucho, porque desconecta entradas.

In [ ]:
# 7.1 Tabla comparativa (3 variantes + referencia)
TABLA = pd.DataFrame(FILAS + [FILA_REF]).set_index("variante")
TABLA["subida val entropía"] = TABLA["val entropía final"] - TABLA["val entropía mín."]
mostrar = TABLA[["mejor época", "train acc", "val acc", "brecha train−val", "val entropía mín.", "subida val entropía",
                 "test acc", "test top-3", "test F1 macro", "pesos ≈ 0 (|w|<1e-3)", "norma L2 de los pesos"]]
TABLA.to_csv(EVI / "tabla_comparativa_regularizacion.csv", encoding="utf-8-sig")
pct = ["train acc", "val acc", "brecha train−val", "test acc", "test top-3", "pesos ≈ 0 (|w|<1e-3)"]
display(mostrar.style.format({**{c: "{:.2%}" for c in pct}, "val entropía mín.": "{:.4f}", "subida val entropía": "{:+.4f}",
                              "test F1 macro": "{:.4f}", "norma L2 de los pesos": "{:.1f}", "mejor época": "{:.0f}"})
                .highlight_max(subset=["val acc", "test acc", "test F1 macro"], color="#d4edda")
                .highlight_min(subset=["brecha train−val"], color="#d4edda"))

---
# 8. Sensibilidad a λ y decisión
Antes de fijar λ = 0,0001 se prueba un valor 10 veces menor y uno 10 veces mayor, para L1 y para L2.
Si λ es muy pequeño, la penalización casi no actúa. Si es muy grande, el modelo pierde capacidad y la exactitud cae.

In [ ]:
# 8.1 Barrido de λ (mismo Dropout, misma semilla)
SWEEP = []
for tipo, Reg in [("L2", regularizers.L2), ("L1", regularizers.L1)]:
    for lam in [1e-5, 1e-4, 1e-3]:
        if lam == LAMBDA:
            f = next(x for x in FILAS if x["variante"] == f"Dropout + {tipo}")
        else:
            f, _, _ = entrenar(f"Dropout + {tipo} (λ={lam:g})", Reg(lam))
        SWEEP.append({"tipo": tipo, "λ": lam, "val acc": f["val acc"], "brecha train−val": f["brecha train−val"],
                      "pesos ≈ 0": f["pesos ≈ 0 (|w|<1e-3)"]})
SWEEP = pd.DataFrame(SWEEP)
SWEEP.to_csv(EVI / "barrido_lambda.csv", index=False, encoding="utf-8-sig")
display(SWEEP.pivot(index="λ", columns="tipo", values="val acc").style.format("{:.2%}").highlight_max(axis=None, color="#d4edda"))

fig, ax = plt.subplots(figsize=(7, 4))
for tipo, c in [("L2", "#1f77b4"), ("L1", "#d62728")]:
    s = SWEEP[SWEEP.tipo == tipo]; ax.plot(s["λ"], s["val acc"] * 100, "o-", color=c, lw=2, label=f"Dropout + {tipo}")
ax.axhline(TABLA.loc["Solo Dropout", "val acc"] * 100, color="#2ca02c", ls="--", label="Solo Dropout")
ax.set_xscale("log"); ax.set_xlabel("λ"); ax.set_ylabel("exactitud de validación (%)"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("Exactitud de validación según λ"); plt.tight_layout(); plt.savefig(EVI / "barrido_lambda.png", dpi=120); plt.show()

In [ ]:
# 8.2 Decisión con la regla fijada al inicio (validación; desempate por menor sobreajuste)
tres = TABLA.loc[[n for n, _ in VARIANTES]]
mejor_val = tres["val acc"].max()
empatadas = tres[tres["val acc"] >= mejor_val - 0.005]
ELEGIDA = empatadas["brecha train−val"].idxmin() if len(empatadas) > 1 else tres["val acc"].idxmax()
DESCARTADAS = [n for n in tres.index if n != ELEGIDA]
e = TABLA.loc[ELEGIDA]

lineas = [f"DECISIÓN: {ELEGIDA} (λ = {LAMBDA:g})",
          f"  · Validación {e['val acc']:.2%} | prueba {e['test acc']:.2%} | F1 macro {e['test F1 macro']:.3f} | "
          f"brecha train−val {e['brecha train−val']:.2%}"]
if len(empatadas) > 1:
    lineas.append(f"  · {', '.join(empatadas.index)} quedaron a menos de 0,5 puntos en validación: "
                  f"se desempata por la menor brecha (menos sobreajuste).")
for n in DESCARTADAS:
    d = TABLA.loc[n]
    lineas.append(f"  · Se descarta {n}: validación {d['val acc']:.2%} ({(d['val acc'] - e['val acc']) * 100:+.2f} pts), "
                  f"brecha {d['brecha train−val']:.2%}, pesos ≈ 0 {d['pesos ≈ 0 (|w|<1e-3)']:.1%}")
r = TABLA.loc[REF]
sd = TABLA.loc["Solo Dropout"]
lineas.append(f"  · Referencia sin Dropout ni L1/L2: brecha {r['brecha train−val']:.2%} (con Dropout {sd['brecha train−val']:.2%}) "
              f"y la entropía de validación sube {r['subida val entropía']:+.3f} al final (con Dropout {sd['subida val entropía']:+.3f})"
              + (" → el Dropout reduce el sobreajuste." if r['brecha train−val'] > sd['brecha train−val'] else "."))
l1 = TABLA.loc["Dropout + L1"]
if ELEGIDA != "Dropout + L1":
    lineas.append(f"  · L1 apaga el {l1['pesos ≈ 0 (|w|<1e-3)']:.0%} de los pesos y no mejora la validación: "
                  f"las 960 características de la red base aportan información, y desconectarlas no ayuda.")
else:
    lineas.append(f"  · L1 apaga el {l1['pesos ≈ 0 (|w|<1e-3)']:.0%} de los pesos y aun así obtiene la mejor validación.")
DECISION_TXT = "\n".join(lineas)
log.info("=" * 70)
for l in lineas: log.info(l)
(EVI / "decision_regularizacion.txt").write_text(DECISION_TXT, encoding="utf-8")

---
# 9. Tabla de decisiones técnicas (PDF de 1 página)
Las tres primeras filas salen de **este** cuaderno. Las demás vienen de los cuadernos del proyecto ya ejecutados en Kaggle (entrenamiento, ensamble y destilación), con la misma división de datos.

In [ ]:
# 9.1 Genera tabla_decisiones_tecnicas.pdf
import textwrap
from matplotlib.backends.backend_pdf import PdfPages

pp = lambda x: f"{x * 100:.1f} %".replace(".", ",")
dec = lambda x: f"{x:+.3f}".replace(".", ",")
e, r = TABLA.loc[ELEGIDA], TABLA.loc[REF]
sd = TABLA.loc["Solo Dropout"]
fam = ELEGIDA.split("+ ")[-1] if "+" in ELEGIDA else "L2"
sw = SWEEP[SWEEP.tipo == fam].set_index("λ")["val acc"]
otras = "; ".join(f"{n}: {pp(TABLA.loc[n, 'val acc'])} val, brecha {pp(TABLA.loc[n, 'brecha train−val'])}"
                  + (f", {TABLA.loc[n, 'pesos ≈ 0 (|w|<1e-3)']:.0%} de pesos en cero" if "L1" in n else "")
                  for n in DESCARTADAS)

FILAS_PDF = [
    ("Regularización", f"{ELEGIDA} (λ = 0,0001)", " / ".join(DESCARTADAS),
     f"{ELEGIDA}: {pp(e['val acc'])} val, {pp(e['test acc'])} prueba, brecha {pp(e['brecha train−val'])}. {otras}."),
    ("Dropout en la cabeza", "Dropout 0,4 y 0,3", "Sin Dropout",
     f"Brecha train−val: {pp(r['brecha train−val'])} sin Dropout vs. {pp(sd['brecha train−val'])} con Dropout. "
     f"La entropía de validación sube {dec(r['subida val entropía'])} al final sin Dropout y {dec(sd['subida val entropía'])} con Dropout."),
    ("Fuerza de la penalización", "λ = 0,0001", "λ = 0,00001 / λ = 0,001",
     f"Validación con {fam}: {pp(sw.get(1e-5, np.nan))} (λ=1e-5), {pp(sw.get(1e-4, np.nan))} (λ=1e-4), {pp(sw.get(1e-3, np.nan))} (λ=1e-3)."),
    ("Red base y ajuste fino", "EfficientNetV2-S 260 px con ajuste fino (60 % de la base)", "Red base congelada (solo cabeza)",
     f"92,6 % val / 93,1 % prueba (EfficientNetV2-S, época 39) vs. {pp(sd['val acc'])} val con base congelada en este experimento."),
    ("Ensamble", "2 EfficientNetV2-S con pesos 0,4 / 0,6 + TTA", "Un solo modelo (v4)",
     "Prueba 93,73 % vs. 93,12 %; F1 macro 0,920 vs. 0,913; top-3 98,49 % vs. 98,21 %."),
    ("Ubicación", "Ajuste por departamento (α = 1, λ = 1, elegidos con validación)", "Solo la imagen",
     "Prueba 95,36 % vs. 93,73 %; F1 macro 0,941 vs. 0,920. Cumple la meta de 94 %."),
    ("Modelo para el celular", "Destilar a MobileNetV3-Large (T = 4, α = 0,3)", "Usar el ensamble en el celular",
     "630,5 MB → 6,1 MB y 2.357,8 ms → 3,7 ms por foto; acierto con ubicación 95,4 % → 93,7 %."),
    ("Formato TFLite", "float16", "int8 dinámico / float32",
     "float16: 6,1 MB, 3,7 ms, 93,7 %. int8: 3,3 MB, 15,5 ms, 93,1 %. float32: 12,0 MB, mismo acierto."),
]

def envolver(t, ancho):
    return "\n".join(textwrap.wrap(t, ancho))

ruta_pdf = SALIDA / "tabla_decisiones_tecnicas.pdf"
COLW = [0.12, 0.22, 0.17, 0.49]; CHARS = [15, 30, 23, 80]
with PdfPages(ruta_pdf) as pdf:
    fig = plt.figure(figsize=(11.69, 8.27))                       # A4 horizontal
    fig.text(0.04, 0.945, "Tabla de decisiones técnicas — Mamíferos de Colombia", fontsize=17, weight="bold", color="#1b4332")
    fig.text(0.04, 0.915, "Clasificación de 40 mamíferos en fotos de cámaras trampa · Jaime Andrés Ortega Cadena · "
             "David Alejandro Muñoz Carmona", fontsize=9.5, color="#444444")
    x0, y = 0.04, 0.875; ancho = 0.92
    xs = [x0 + sum(COLW[:i]) * ancho for i in range(4)]
    fig.patches.append(plt.Rectangle((x0, y - 0.035), ancho, 0.035, transform=fig.transFigure, color="#1b4332"))
    for x, t in zip(xs, ["Aspecto", "Decisión tomada", "Alternativa descartada", "Evidencia numérica que la respalda"]):
        fig.text(x + 0.006, y - 0.0235, t, fontsize=10, weight="bold", color="white")
    y -= 0.035
    for i, fila in enumerate(FILAS_PDF):
        textos = [envolver(t, c) for t, c in zip(fila, CHARS)]
        n = max(t.count("\n") + 1 for t in textos); alto = 0.0215 * n + 0.02
        if i % 2 == 0:
            fig.patches.append(plt.Rectangle((x0, y - alto), ancho, alto, transform=fig.transFigure, color="#eef4ef"))
        for j, (x, t) in enumerate(zip(xs, textos)):
            fig.text(x + 0.006, y - 0.01, t, fontsize=9.2, va="top", linespacing=1.35,
                     weight="bold" if j in (0, 1) else "normal", color="#1b4332" if j == 1 else "#222222")
        y -= alto
    fig.text(0.04, max(y - 0.03, 0.035),
             envolver("Fuente: filas 1–3 de este cuaderno (cabeza Dropout 0,4 → Densa 512 → Dropout 0,3 sobre MobileNetV3-Large congelada, "
                      "misma semilla). Filas 4–8: cuadernos del proyecto ejecutados en Kaggle. Misma división por registro GBIF: "
                      "17.960 entrenamiento · 2.424 validación · 2.458 prueba (solo fotos de Colombia en validación y prueba).", 190),
             fontsize=7.6, color="#555555", va="top", linespacing=1.3)
    pdf.savefig(fig); plt.close(fig)
log.info(f"PDF listo: {ruta_pdf}")
from IPython.display import FileLink; display(FileLink(str(ruta_pdf.relative_to(BASE_SALIDA)) if EN_KAGGLE else str(ruta_pdf)))

---
# Conclusión
- Las tres variantes usan la misma cabeza, los mismos datos y el mismo λ; la única diferencia es la penalización de los pesos.
- La decisión se toma con **validación** y se confirma con **prueba** (sección 8.2).
- **L1** empuja muchos pesos a cero (modelo disperso); **L2** los mantiene activos pero pequeños. La tabla de la sección 7 muestra cuál de los dos comportamientos le sirve más a este problema.
- El texto de la decisión (8.2) y el PDF (9.1) se generan solos con los números de esta ejecución.
- Los archivos de evidencia quedan en `regularizacion/evidencia/`: curvas, tabla comparativa, barrido de λ y decisión. El PDF queda en `regularizacion/tabla_decisiones_tecnicas.pdf`.